In [1]:
import glob
import matplotlib.pyplot as plt
import numpy as np
import os
from plot_progress import gather_metrics
from parse_levels import find_levels_in_configs, find_levels_in_configs_glob
from parse_levels import process_metrics, human_train_time_dict, convert_to_dict, compute_gap_in_percentage, compute_gap_in_percentage_list, convert_to_dict_multiple_runs
from parse_levels import filter_folder_info
import pandas as pd
import matplotlib.ticker as ticker
from plot_utils import plot_gap_comparison
import re
import copy
import json

In [2]:
# for each item in the dict, if any two have the same 'record', remove the one with lower number of steps in metric
# def deduplicate_metrics(search_results):
#     records_length_so_far = {}
#     new_search_results = {}
#     for key, value in search_results.items():
#         if value['record'] not in records_length_so_far:
#             records_length_so_far[value['record']] = (len(value['metrics']['step']), key)
#             new_search_results[key] = value
#         else:
#             if len(value['metrics']['step']) > records_length_so_far[value['record']][0]:
#                 new_search_results.pop(records_length_so_far[value['record']][1])

#                 records_length_so_far[value['record']] = (len(value['metrics']['step']), key)
#                 new_search_results[key] = value
#     return new_search_results



In [3]:
ori_results = find_levels_in_configs_glob(
    [
        '/checkpoint/maui/zhaobc/scientist/workspace/record_*',
        # '/checkpoint/maui/zhaobc/scientist/workspace/record_*_20250424_*',
        # '/checkpoint/maui/zhaobc/scientist/workspace/record_*_20250425_*',
        # '/checkpoint/maui/zhaobc/scientist/workspace/record_*_20250405_*'
        # '/checkpoint/maui/zhaobc/scientist/workspace/record_1[2-8]_20250412_*',
        #  '/checkpoint/maui/zhaobc/scientist/workspace/record_*_20250408_*',
        # '/checkpoint/maui/zhaobc/scientist/workspace/record_*_20250405_*'
        # '/checkpoint/maui/zhaobc/scientist/workspace/record_1[2-8]_20250409_*',
    ]
)
# with open('/checkpoint/maui/zhaobc/scientist/code_analysis_with_all_versions_knowledge_o3_mini.json', 'r') as f:
#     o3_results = json.load(f)


Found 2401 directories


  0%|          | 0/2401 [00:00<?, ?it/s]

100%|██████████| 2401/2401 [17:44<00:00,  2.25it/s] 


In [4]:
# with open('may8.cache', 'w') as f:
#     json.dump(ori_results, f)

In [3]:
with open('may8.cache', 'r') as f:
    ori_results = json.load(f)

In [4]:
# z_folder_info = filter_folder_info(ori_results, [('levels', 'z')])
unique_levels = set()
for key, value in ori_results.items():
    unique_levels.add(value['levels'][0])
unique_levels

{1, 12, 125, 129, 19, 2, 5, 'z'}

In [13]:
# 1 is psuedo-code
# 2 is text description
# 5 is paper like 
# 9 is adhoc knowledge, we can ignore it for now
# 'z' is zero knowledge

In [5]:
folder_info = ori_results
# o3_results

In [6]:
len(folder_info)

2378

In [ ]:
# flat search -- n_initial_hypotheses = 50
flat_params = [
    ('runner', 'bon'),
    # ('n_initial_hypotheses', 50),
    ('n_iterations', 1),
]
# tree search -- n_initial_hypotheses = 1, n_hypotheses = 3
tree_params = [
    ('runner', 'bon'),
    ('n_initial_hypotheses', 1),
    ('n_hypotheses', 3),
]
# forest search -- n_initial_hypotheses = 3, n_hypotheses = 3
forest_params = [
    ('runner', 'bon'),
    ('n_initial_hypotheses', 3),
    ('n_hypotheses', 3),
]
# AIDE -- n_initial_hypotheses = 3, n_hypotheses = 1, debug_prob = 0.5
aide_params = [
    ('runner', 'aide'),
    ('n_initial_hypotheses', 3),
    ('n_hypotheses', 1),
    ('debug_prob', 0.5),
]
# MultiAIDE -- n_initial_hypotheses = 3, n_hypotheses = 3, debug_prob = 0.5
multi_aide_params = [
    ('runner', 'aide'),
    ('n_initial_hypotheses', 3),
    ('n_hypotheses', 3),
    ('debug_prob', 0.5),
]

search_algo_params = {
    'flat': flat_params,
    'tree': tree_params,
    'forest': forest_params,
    'ori_aide': aide_params,
    'multi_aide': multi_aide_params,
}

plot_info = {}

for search_algo, params in search_algo_params.items():
    plot_info[search_algo] = {}
    for level in [1, 12, 125, 'z']:
        plot_info[search_algo][level] = {}
        # for model in ['deepseek-r1', 'o3-mini']:
        for model in ['o3-mini']:
            search_params = params + [('levels', level), ('model', model)]
            filtered_folder_info = filter_folder_info(folder_info, search_params)
            print(f'{search_algo} {level} {model} {len(filtered_folder_info)}')
            plot_info[search_algo][level][model] = filtered_folder_info



flat 1 o3-mini 31
flat 12 o3-mini 59
flat 125 o3-mini 55
flat z o3-mini 20
tree 1 o3-mini 18
tree 12 o3-mini 46
tree 125 o3-mini 55
tree z o3-mini 20
forest 1 o3-mini 18
forest 12 o3-mini 40
forest 125 o3-mini 70
forest z o3-mini 20
ori_aide 1 o3-mini 15
ori_aide 12 o3-mini 30
ori_aide 125 o3-mini 37
ori_aide z o3-mini 20
multi_aide 1 o3-mini 18
multi_aide 12 o3-mini 55
multi_aide 125 o3-mini 103
multi_aide z o3-mini 20


In [8]:
figure_data = {}
for search_algo in search_algo_params.keys():
    for level in [1, 12, 125, 'z']:
        for model in ['o3-mini']:
            plot_info[search_algo][level][model] = process_metrics(plot_info[search_algo][level][model])
            # plot_info[search_algo][level][model] = deduplicate_metrics(plot_info[search_algo][level][model])
            figure_data[f'{search_algo}_{level}_{model}'] = convert_to_dict_multiple_runs(plot_info[search_algo][level][model])


In [9]:
human_train_time_dict = {
    1: 2968348,
    2: 2209926,
    3: 1386147,
    4: 1301740,
    5: 949528,
    6: 766259,
    7: 773072,
    8: 662205,
    9: 505531,
    10: 477150,
    11: 442985,
    12: 317839,
    13: 289805,
    14: 273107,
    15: 241463,
    16: 232971,
    17: 220374,
    18: 211840,
    19: 199442,
    20: 188680,
    21: 184262
}


In [ ]:
figure_data_percent = {}
for key in figure_data.keys():
    print(key)
    figure_data_percent[key] = compute_gap_in_percentage_list(figure_data[key], human_train_time_dict)
    figure_data_percent[key] = {str(k): v for k, v in figure_data_percent[key].items()}
    del figure_data_percent[key]['6']

flat_1_o3-mini
flat_12_o3-mini
flat_125_o3-mini
flat_z_o3-mini
tree_1_o3-mini
tree_12_o3-mini
tree_125_o3-mini
tree_z_o3-mini
forest_1_o3-mini
forest_12_o3-mini
forest_125_o3-mini
forest_z_o3-mini
ori_aide_1_o3-mini
ori_aide_12_o3-mini
ori_aide_125_o3-mini
ori_aide_z_o3-mini
multi_aide_1_o3-mini
multi_aide_12_o3-mini
multi_aide_125_o3-mini
multi_aide_z_o3-mini


In [11]:
# replace the ones with 300% improvement with 0 as they might be summarizer mistakes
for key in figure_data_percent.keys():
    for k, vs in figure_data_percent[key].items():
        for i, v in enumerate(vs):
            if v > 3 or v < 0:
                print(f'found {v} in {key} {k}')
                figure_data_percent[key][k][i] = 0.

found 13.95647592565491 in forest_12_o3-mini 10
found 18.531871399368146 in multi_aide_12_o3-mini 19
found 17.802085902540433 in multi_aide_125_o3-mini 9


In [ ]:
figure_data_percent # this is the training time reduction in percentage, the numpy array contains the percentage for each run

{'flat_1_o3-mini': {'15': array([0.38577485, 0.        ]),
  '8': array([0., 0.]),
  '17': array([0.38856339, 0.        ]),
  '5': array([0., 0.]),
  '11': array([0.        , 0.        , 0.06942291]),
  '16': array([0.        , 0.34960705]),
  '12': array([0.17856888, 0.        ]),
  '9': array([1.15232021, 0.30108171]),
  '13': array([0.        , 0.01868487]),
  '7': array([0.        , 0.68088791]),
  '14': array([0., 0.]),
  '18': array([0.       , 0.1525246]),
  '4': array([0.        , 0.03780394]),
  '10': array([0.        , 0.85909557])},
 'flat_12_o3-mini': {'11': array([0.        , 0.04983779, 0.07909162, 0.        , 0.        ]),
  '2': array([0.]),
  '12': array([0.        , 0.17821217, 0.13779696, 0.        ]),
  '17': array([0.14565268, 0.        , 0.50843684, 0.        ]),
  '7': array([0.60557244, 0.68070751, 0.62263794]),
  '15': array([0.37988695, 0.        , 0.45949129, 0.        ]),
  '16': array([0.       , 0.       , 0.2943558, 0.       ]),
  '18': array([0.        ,

In [14]:
# compute pass@50% for everything
pass_at_50 = {}
for key in figure_data_percent.keys():
    pass_at_50[key] = {}
    for k, vs in figure_data_percent[key].items():
        pass_at_50[key][k] = np.mean(vs > 0.5)
pass_at_50


{'flat_1_o3-mini': {'15': 0.0,
  '8': 0.0,
  '17': 0.0,
  '5': 0.0,
  '11': 0.0,
  '16': 0.0,
  '12': 0.0,
  '9': 0.5,
  '13': 0.0,
  '7': 0.5,
  '14': 0.0,
  '18': 0.0,
  '4': 0.0,
  '10': 0.5},
 'flat_12_o3-mini': {'11': 0.0,
  '2': 0.0,
  '12': 0.0,
  '17': 0.25,
  '7': 1.0,
  '15': 0.0,
  '16': 0.0,
  '18': 0.0,
  '13': 0.0,
  '5': 0.3333333333333333,
  '14': 0.0,
  '10': 0.6666666666666666,
  '9': 0.6666666666666666,
  '19': 0.0,
  '8': 0.0,
  '1': 1.0,
  '4': 0.3333333333333333,
  '3': 0.0},
 'flat_125_o3-mini': {'17': 0.0,
  '4': 0.6666666666666666,
  '7': 0.3333333333333333,
  '12': 0.3333333333333333,
  '5': 0.6666666666666666,
  '13': 0.0,
  '18': 0.0,
  '15': 0.0,
  '16': 0.0,
  '1': 0.0,
  '10': 0.6666666666666666,
  '3': 0.0,
  '11': 0.0,
  '14': 0.0,
  '2': 0.0,
  '8': 0.0,
  '19': 0.5,
  '9': 0.6666666666666666},
 'flat_z_o3-mini': {'7': 0.0,
  '13': 0.0,
  '1': 1.0,
  '3': 0.0,
  '11': 0.0,
  '19': 0.0,
  '14': 0.0,
  '16': 0.0,
  '5': 0.0,
  '17': 0.0,
  '8': 0.0,
  '1

In [16]:
# get colors for the two models
# colors = ['crimson', 'royalblue']
# for search_algo in ['flat', 'tree', 'forest', 'ori_aide', 'multi_aide']:
#     filtered_figure_data = {k: v for k, v in figure_data_percent.items() if search_algo in k}
#     data_dicts = []
    
#     # For each hint level, add both models
#     for level in [1, 12, 125, 'z']:
#         if level == 125:
#             level_plot = 123
#         else:
#             level_plot = level
        
#         # Add o3-mini model data
#         data_dicts.append((
#             filtered_figure_data[f'{search_algo}_{level}_o3-mini'],
#             f'{level}',
#             colors[1]
#         ))

#     plot_gap_comparison(
#         data_dicts,
#         figsize=(15, 10),  # Reduced height since we only need 1 row
#         n_cols=4,
#         main_title=f'Recovered Time Gap Comparison: {search_algo}'
#     )
#     plt.show()
